# LeetCode 2404 — Most Frequent Even Element bằng Python

Link đề: https://leetcode.com/problems/most-frequent-even-element/ · Độ khó: **Easy** · Chủ đề: Array, Hash Table, Counting

Skeleton LeetCode:

```python
class Solution:
    def mostFrequentEven(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **đếm tần suất có điều kiện**: chỉ quan tâm số **chẵn**, chọn số xuất hiện nhiều nhất, hòa thì chọn số **nhỏ nhất**, không có số chẵn thì trả `-1`.
2. Dùng toán tử `%` để kiểm tra chẵn/lẻ và `collections.Counter` để đếm tần suất.
3. Biết cách xử lý **tie-break** (quy tắc phá hòa) bằng so sánh `(đếm, -giá trị)` hoặc bằng điều kiện `count > best_count or (count == best_count and value < best)`.
4. Viết 4 cách: đếm lại bằng `nums.count` (brute), `Counter` một lượt, sắp xếp rồi đếm đoạn liên tiếp, và mảng đếm theo giá trị (vì `0 <= nums[i] <= 10^5`); đối chiếu bằng test ngẫu nhiên.
5. Tránh các lỗi: hòa mà chọn số lớn, trả về `0` thay vì `-1`, đếm cả số lẻ, trả về tần suất thay vì giá trị, và tin `Counter.most_common` sẽ phá hòa theo giá trị nhỏ nhất.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Trả về **phần tử chẵn xuất hiện nhiều nhất**. Nếu có nhiều phần tử chẵn cùng số lần xuất hiện nhiều nhất, trả về phần tử **nhỏ nhất** trong số đó. Nếu **không có** phần tử chẵn nào, trả về `-1`.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[0,1,2,2,4,4,1]` | `2` | chẵn: `0`(1 lần), `2`(2 lần), `4`(2 lần); hòa giữa 2 và 4, chọn nhỏ nhất `2` |
| 2 | `[4,4,4,9,2,4]` | `4` | `4` xuất hiện 4 lần, `2` xuất hiện 1 lần |
| 3 | `[29,47,21,41,13,37,25,7]` | `-1` | không có số chẵn |

Constraints: `1 <= nums.length <= 2000`; `0 <= nums[i] <= 10^5`.

In [1]:
nums = [0, 1, 2, 2, 4, 4, 1]
evens = [x for x in nums if x % 2 == 0]
print("các số chẵn:", evens)
for v in sorted(set(evens)):
    print(f"  {v} xuất hiện {evens.count(v)} lần")

các số chẵn: [0, 2, 2, 4, 4]
  0 xuất hiện 1 lần
  2 xuất hiện 2 lần
  4 xuất hiện 2 lần


## 3. Kiến thức nền

### 3.1. Số chẵn và toán tử `%`

`x % 2` là phần dư khi chia cho 2: `0` nếu `x` chẵn, `1` nếu lẻ. Số `0` **là số chẵn**; đề cho phép `nums[i] = 0` nên phải coi `0` là một ứng viên.

In [2]:
for x in (0, 1, 2, 7, 100000):
    print(x, "chẵn" if x % 2 == 0 else "lẻ")

0 chẵn
1 lẻ
2 chẵn
7 lẻ
100000 chẵn


### 3.2. `collections.Counter` để đếm tần suất

`Counter(iterable)` trả về dict từ phần tử sang số lần xuất hiện. `Counter` của phần tử không có trả về `0` thay vì lỗi. `c.items()` duyệt cặp `(giá trị, số lần)`.

In [3]:
from collections import Counter
c = Counter([4, 4, 4, 9, 2, 4])
print(c)
print(c[4], c[2], c[100])
print(list(c.items()))

Counter({4: 4, 9: 1, 2: 1})
4 1 0
[(4, 4), (9, 1), (2, 1)]


### 3.3. Khóa so sánh `(đếm, -giá trị)` để phá hòa

`max(items, key=...)` chọn phần tử có khóa lớn nhất. Tuple so sánh **từ trái sang phải**: so đếm trước, hòa thì so tiếp. Dùng `-giá trị` để "giá trị nhỏ hơn thì khóa lớn hơn", tức chọn số nhỏ nhất khi hòa.

In [4]:
pairs = [(0, 1), (2, 2), (4, 2)]          # (giá trị, số lần)
best = max(pairs, key=lambda p: (p[1], -p[0]))
print(best)                                 # (2, 2): hòa đếm, giá trị 2 nhỏ hơn 4
print((2, -2) > (2, -4))

(2, 2)
True


### 3.4. `Counter.most_common` KHÔNG phá hòa theo giá trị nhỏ nhất

`most_common(1)` trả về phần tử có đếm lớn nhất; nếu hòa thì giữ thứ tự **xuất hiện đầu tiên** (thứ tự chèn của dict). Vì vậy nó không phù hợp để phá hòa theo "giá trị nhỏ nhất".

In [5]:
from collections import Counter
c = Counter([4, 4, 2, 2])
print(c.most_common(1))     # [(4, 2)]: 4 xuất hiện trước, dù 2 nhỏ hơn
print(min(k for k, v in c.items() if v == max(c.values())))

[(4, 2)]
2


### 3.5. Mảng đếm (counting array)

Giá trị chỉ tới `10^5` nên có thể dùng mảng độ dài `100001`, `cnt[v] += 1`. Duyệt `v` tăng dần và cập nhật khi `cnt[v] > best_count` (chỉ khi **lớn hơn hẳn**) sẽ tự giữ số nhỏ nhất khi hòa.

In [6]:
cnt = [0] * 11
for x in [0, 1, 2, 2, 4, 4, 1]:
    cnt[x] += 1
print(cnt)
best_v, best_c = -1, 0
for v in range(0, 11, 2):      # chỉ duyệt số chẵn, tăng dần
    if cnt[v] > best_c:
        best_v, best_c = v, cnt[v]
print(best_v, best_c)

[1, 2, 2, 0, 2, 0, 0, 0, 0, 0, 0]
2 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay ví dụ 1, `nums = [0,1,2,2,4,4,1]`:

1. Gạch bỏ số lẻ (`1`, `1`): còn `[0, 2, 2, 4, 4]`.
2. Đếm: `0 -> 1`, `2 -> 2`, `4 -> 2`.
3. Số lần lớn nhất là `2`. Hai số cùng đạt: `2` và `4`.
4. Hòa thì chọn số nhỏ nhất ⇒ `2`.

Ví dụ 3: sau khi gạch bỏ số lẻ không còn gì ⇒ `-1`.

Như vậy bài chia thành ba việc nhỏ: **(a) lọc số chẵn, (b) đếm, (c) chọn theo (đếm lớn nhất, giá trị nhỏ nhất)**. Chú ý đừng nhầm: kết quả là **giá trị**, không phải số lần.

In [7]:
nums = [0, 1, 2, 2, 4, 4, 1]
evens = [x for x in nums if x % 2 == 0]
print("sau khi lọc:", evens)
counts = {}
for v in evens:
    counts[v] = counts.get(v, 0) + 1
print("đếm:", counts)
top = max(counts.values()) if counts else 0
print("số lần lớn nhất:", top, "-> ứng viên:", [v for v, c in counts.items() if c == top])
print("đáp án:", min(v for v, c in counts.items() if c == top) if counts else -1)

sau khi lọc: [0, 2, 2, 4, 4]
đếm: {0: 1, 2: 2, 4: 2}
số lần lớn nhất: 2 -> ứng viên: [2, 4]
đáp án: 2


## 5. Cách 1 — Brute Force / baseline: đếm lại bằng `nums.count`

**Ý tưởng:** với mỗi giá trị chẵn **phân biệt** (lấy từ `set`), gọi `nums.count(v)` để đếm. Duyệt qua các ứng viên, cập nhật đáp án khi đếm lớn hơn, hoặc bằng và giá trị nhỏ hơn.

In [8]:
from typing import List

class SolutionBrute:
    def mostFrequentEven(self, nums: List[int]) -> int:
        best_val, best_cnt = -1, 0
        for v in set(nums):
            if v % 2 != 0:
                continue
            c = nums.count(v)                       # O(n) cho mỗi giá trị
            if c > best_cnt or (c == best_cnt and v < best_val):
                best_val, best_cnt = v, c
        return best_val

In [9]:
sol = SolutionBrute()
print(sol.mostFrequentEven([0, 1, 2, 2, 4, 4, 1]))             # 2
print(sol.mostFrequentEven([4, 4, 4, 9, 2, 4]))                # 4
print(sol.mostFrequentEven([29, 47, 21, 41, 13, 37, 25, 7]))   # -1

2
4
-1


**Độ phức tạp:** `set(nums)` có tối đa `n` giá trị, mỗi lần `nums.count` mất `O(n)` ⇒ Time `O(n^2)` trong trường hợp xấu (mọi số chẵn khác nhau). Space `O(n)` cho `set`. **Nút thắt:** đếm lại cả mảng cho mỗi giá trị; nên đếm tất cả trong **một** lượt.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** đếm tất cả bằng một lượt qua mảng (`Counter` hoặc dict), sau đó chọn đáp án.

- **Invariant khi duyệt các cặp `(v, c)`:** `(best_val, best_cnt)` luôn là đáp án tốt nhất trong các cặp đã xét, theo thứ tự ưu tiên *đếm lớn hơn*, rồi *giá trị nhỏ hơn*.
- **Quy tắc cập nhật:**
  - **nếu** `c > best_cnt` **thì** lấy `(v, c)`;
  - **nếu** `c == best_cnt` **và** `v < best_val` **thì** cũng lấy `(v, c)`;
  - ngược lại bỏ qua.
- Khởi tạo `best_val = -1`, `best_cnt = 0` để khi không có số chẵn nào, kết quả giữ nguyên `-1`. Vì đếm của một số xuất hiện luôn `>= 1 > 0`, số chẵn đầu tiên luôn được chọn.

## 7. Các cách tiếp theo

### Cách 2 — `Counter` một lượt

Chỉ đếm số chẵn rồi duyệt các cặp với quy tắc ở mục 6.

In [10]:
from typing import List
from collections import Counter

class SolutionCounter:
    def mostFrequentEven(self, nums: List[int]) -> int:
        cnt = Counter(x for x in nums if x % 2 == 0)
        best_val, best_cnt = -1, 0
        for v, c in cnt.items():
            if c > best_cnt or (c == best_cnt and v < best_val):
                best_val, best_cnt = v, c
        return best_val

In [11]:
print(SolutionCounter().mostFrequentEven([0, 1, 2, 2, 4, 4, 1]))             # 2
print(SolutionCounter().mostFrequentEven([4, 4, 4, 9, 2, 4]))                # 4
print(SolutionCounter().mostFrequentEven([29, 47, 21, 41, 13, 37, 25, 7]))   # -1

2
4
-1


**Độ phức tạp:** Time `O(n)` (một lượt đếm, một lượt qua các giá trị phân biệt), Space `O(k)` với `k` là số giá trị chẵn phân biệt (`k <= n`).

### Cách 3 — Sắp xếp rồi đếm các đoạn liên tiếp

Lọc số chẵn và sắp xếp tăng dần. Các giá trị bằng nhau đứng liền nhau; đếm độ dài mỗi đoạn. Duyệt tăng dần nên khi hòa ta chỉ cập nhật nếu đoạn **dài hơn hẳn** (giữ giá trị nhỏ hơn).

In [12]:
from typing import List

class SolutionSort:
    def mostFrequentEven(self, nums: List[int]) -> int:
        evens = sorted(x for x in nums if x % 2 == 0)
        best_val, best_cnt = -1, 0
        i = 0
        while i < len(evens):
            j = i
            while j < len(evens) and evens[j] == evens[i]:
                j += 1
            if j - i > best_cnt:                    # lớn hơn hẳn mới cập nhật
                best_val, best_cnt = evens[i], j - i
            i = j
        return best_val

In [13]:
print(SolutionSort().mostFrequentEven([0, 1, 2, 2, 4, 4, 1]))             # 2
print(SolutionSort().mostFrequentEven([4, 4, 4, 9, 2, 4]))                # 4
print(SolutionSort().mostFrequentEven([29, 47, 21, 41, 13, 37, 25, 7]))   # -1

2
4
-1


**Độ phức tạp:** Time `O(n log n)` do sắp xếp, Space `O(n)` cho danh sách chẵn đã lọc.

### Cách 4 — Mảng đếm theo giá trị

Vì `0 <= nums[i] <= 10^5`, dùng mảng `cnt` kích thước `100001`. Duyệt `v = 0, 2, 4, ...` tăng dần, cập nhật khi `cnt[v] > best_cnt`.

In [14]:
from typing import List

class SolutionArray:
    def mostFrequentEven(self, nums: List[int]) -> int:
        cnt = [0] * (10**5 + 1)
        for x in nums:
            cnt[x] += 1
        best_val, best_cnt = -1, 0
        for v in range(0, 10**5 + 1, 2):
            if cnt[v] > best_cnt:
                best_val, best_cnt = v, cnt[v]
        return best_val

In [15]:
print(SolutionArray().mostFrequentEven([0, 1, 2, 2, 4, 4, 1]))             # 2
print(SolutionArray().mostFrequentEven([4, 4, 4, 9, 2, 4]))                # 4
print(SolutionArray().mostFrequentEven([29, 47, 21, 41, 13, 37, 25, 7]))   # -1

2
4
-1


**Độ phức tạp:** Time `O(n + V)` với `V = 10^5`, Space `O(V)`. Cách này chạy ổn nhờ cận giá trị nhỏ, nhưng cách `Counter` linh hoạt hơn khi giá trị không bị chặn.

## 8. TRACE từng bước

Cách `Counter` trên `nums = [0,1,2,2,4,4,1]`. Sau khi đếm số chẵn: `{0: 1, 2: 2, 4: 2}`.

| Bước | `v` | `c` | `best_val` trước | `best_cnt` trước | Điều kiện | Sau |
|---|---|---|---|---|---|---|
| 1 | 0 | 1 | -1 | 0 | `1 > 0` | `(0, 1)` |
| 2 | 2 | 2 | 0 | 1 | `2 > 1` | `(2, 2)` |
| 3 | 4 | 2 | 2 | 2 | `2 == 2` nhưng `4 < 2` sai | giữ `(2, 2)` |

Đáp án `2`.

In [16]:
def trace_even(nums):
    cnt = {}
    for x in nums:
        if x % 2 == 0:
            cnt[x] = cnt.get(x, 0) + 1
    print("đếm số chẵn:", cnt)
    best_val, best_cnt = -1, 0
    for v, c in cnt.items():
        print(f"xét v={v}, c={c}; hiện tại best=({best_val}, {best_cnt})", end=" -> ")
        if c > best_cnt or (c == best_cnt and v < best_val):
            best_val, best_cnt = v, c
            print("cập nhật")
        else:
            print("giữ nguyên")
    return best_val

print("đáp án:", trace_even([0, 1, 2, 2, 4, 4, 1]))
print("đáp án:", trace_even([29, 47, 21]))

đếm số chẵn: {0: 1, 2: 2, 4: 2}
xét v=0, c=1; hiện tại best=(-1, 0) -> cập nhật
xét v=2, c=2; hiện tại best=(0, 1) -> cập nhật
xét v=4, c=2; hiện tại best=(2, 2) -> giữ nguyên
đáp án: 2
đếm số chẵn: {}
đáp án: -1


## 9. Code nộp LeetCode

Dùng `Counter` một lượt:

In [17]:
from typing import List
from collections import Counter

class Solution:
    def mostFrequentEven(self, nums: List[int]) -> int:
        cnt = Counter(x for x in nums if x % 2 == 0)
        best_val, best_cnt = -1, 0
        for v, c in cnt.items():
            if c > best_cnt or (c == best_cnt and v < best_val):
                best_val, best_cnt = v, c
        return best_val

In [18]:
print(Solution().mostFrequentEven([0, 1, 2, 2, 4, 4, 1]))
print(Solution().mostFrequentEven([4, 4, 4, 9, 2, 4]))
print(Solution().mostFrequentEven([29, 47, 21, 41, 13, 37, 25, 7]))

2
4
-1


## 10. Vì sao thuật toán đúng

1. **Lọc đúng tập ứng viên.** Chỉ các giá trị `v` chẵn xuất hiện trong `nums` mới có thể là đáp án. Nếu tập này rỗng thì không tồn tại đáp án và ta trả `-1`.
2. **Đếm đúng.** `Counter` cho `cnt[v]` bằng đúng số lần `v` xuất hiện.
3. **Invariant của vòng chọn.** Sau khi xét một số cặp, `(best_val, best_cnt)` là phần tử lớn nhất trong các cặp đã xét theo thứ tự `(đếm lớn hơn, rồi giá trị nhỏ hơn)`. Ban đầu (chưa xét gì) `best = (-1, 0)` là "rỗng": mọi cặp thật đều có `c >= 1 > 0` nên sẽ thắng.
4. **Bước cập nhật.** Với cặp `(v, c)`: nếu `c > best_cnt` thì nó tốt hơn hẳn; nếu `c == best_cnt` thì nó tốt hơn chỉ khi `v < best_val`; còn lại không tốt hơn. Như vậy invariant được giữ.
5. **Kết thúc.** Sau khi xét hết, `best_val` là phần tử chẵn xuất hiện nhiều nhất, hòa thì nhỏ nhất: đúng yêu cầu. Các cách còn lại thực hiện cùng phép chọn bằng cách khác (đếm lặp, đoạn liên tiếp theo thứ tự tăng, mảng đếm theo thứ tự tăng).

## 11. Phân tích Time / Space Complexity

Đặt `n = len(nums)`, `k` là số giá trị chẵn phân biệt, `V = 10^5` là giá trị lớn nhất.

| Cách | Time | Space |
|---|---|---|
| Brute (`nums.count`) | `O(n · k)` ⇒ `O(n^2)` xấu nhất | `O(k)` |
| `Counter` | `O(n)` | `O(k)` |
| Sắp xếp | `O(n log n)` | `O(n)` |
| Mảng đếm | `O(n + V)` | `O(V)` |

Với `n <= 2000` mọi cách đều rất nhanh. Cách `Counter` là cân bằng nhất giữa độ ngắn, tốc độ và tính tổng quát.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

In [19]:
sols = [SolutionBrute(), SolutionCounter(), SolutionSort(), SolutionArray(), Solution()]
cases = [
    ([0, 1, 2, 2, 4, 4, 1], 2),                  # ví dụ 1 (hòa -> nhỏ nhất)
    ([4, 4, 4, 9, 2, 4], 4),                     # ví dụ 2
    ([29, 47, 21, 41, 13, 37, 25, 7], -1),       # ví dụ 3 (không có số chẵn)
    ([0], 0),                                     # chỉ có 0 (0 là số chẵn)
    ([1], -1),                                    # một phần tử lẻ
    ([2], 2),                                     # một phần tử chẵn
    ([4, 4, 2, 2], 2),                            # hòa: chọn nhỏ hơn dù 4 xuất hiện trước
    ([1, 1, 1, 2], 2),                            # số lẻ xuất hiện nhiều hơn nhưng bị bỏ qua
    ([100000, 100000, 0, 0, 5], 0),               # hòa ở biên: 0 và 100000
    ([6, 6, 1], 6),                               # trả giá trị 6, không phải tần suất 2
    ([2, 4, 6, 8], 2),                            # tất cả đếm 1 -> nhỏ nhất
    ([0, 0, 3, 3, 3, 3], 0),                      # số lẻ nhiều nhất nhưng đáp án vẫn là 0
]
for nums, want in cases:
    for s in sols:
        assert s.mostFrequentEven(nums) == want, (type(s).__name__, nums, s.mostFrequentEven(nums), want)
print("tất cả test cố định đều đúng với cả 5 cách:", len(cases))

tất cả test cố định đều đúng với cả 5 cách: 12


### 12.2. Đối chiếu ngẫu nhiên (có seed)

Sinh đúng constraints: `1 <= n <= 2000`, `0 <= nums[i] <= 10^5`. Dùng một "oracle" độc lập: sắp xếp các cặp theo `(-đếm, giá trị)` và lấy đầu tiên.

In [20]:
import random
from collections import Counter
random.seed(2404)

def oracle(nums):
    c = Counter(x for x in nums if x % 2 == 0)
    if not c:
        return -1
    return sorted(c.items(), key=lambda p: (-p[1], p[0]))[0][0]

# Giá trị nhỏ -> nhiều hòa
for _ in range(600):
    n = random.randint(1, 12)
    nums = [random.randint(0, 8) for _ in range(n)]
    want = oracle(nums)
    for s in sols:
        assert s.mostFrequentEven(nums) == want, (type(s).__name__, nums)

# Giá trị lớn, n lớn
for _ in range(60):
    n = random.randint(1, 2000)
    nums = [random.randint(0, 10**5) for _ in range(n)]
    want = oracle(nums)
    for s in sols[1:]:                          # bỏ brute vì O(n^2) (vẫn nhanh nhưng không cần)
        assert s.mostFrequentEven(nums) == want
print("600 test nhỏ và 60 test lớn: tất cả cách khớp oracle")

# Toàn số lẻ -> -1
for _ in range(30):
    nums = [2 * random.randint(0, 49999) + 1 for _ in range(random.randint(1, 50))]
    assert all(s.mostFrequentEven(nums) == -1 for s in sols)
print("30 test toàn số lẻ -> -1")

600 test nhỏ và 60 test lớn: tất cả cách khớp oracle
30 test toàn số lẻ -> -1


### 12.3. Phản ví dụ cho tie-break

Hòa nhiều số: đáp án **luôn** là số chẵn nhỏ nhất trong các số đạt đếm lớn nhất.

In [21]:
random.seed(5)
for _ in range(200):
    k = random.randint(2, 5)
    evens = random.sample(range(0, 50, 2), k)
    rep = random.randint(1, 4)
    nums = [e for e in evens for _ in range(rep)] + [random.choice([1, 3, 5]) for _ in range(random.randint(0, 3))]
    random.shuffle(nums)
    assert Solution().mostFrequentEven(nums) == min(evens), (nums, evens)
print("hòa hoàn toàn -> luôn là số chẵn nhỏ nhất")

hòa hoàn toàn -> luôn là số chẵn nhỏ nhất


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Hòa mà chọn số lớn (dùng `>=` thay vì quy tắc phá hòa)

In [22]:
from typing import List
from collections import Counter

def sai_hoa_chon_lon(nums: List[int]) -> int:
    best_val, best_cnt = -1, 0
    for v, c in Counter(x for x in nums if x % 2 == 0).items():
        if c >= best_cnt:                    # SAI: hòa thì luôn ghi đè -> chọn số xuất hiện sau
            best_val, best_cnt = v, c
    return best_val

nums = [2, 2, 4, 4]
print("đúng:", Solution().mostFrequentEven(nums), "| sai:", sai_hoa_chon_lon(nums))
assert Solution().mostFrequentEven(nums) != sai_hoa_chon_lon(nums)

đúng: 2 | sai: 4


**Vì sao sai:** đề yêu cầu số **nhỏ nhất** khi hòa, nhưng `>=` làm cặp đến sau luôn ghi đè nên ra `4`. Hãy dùng `>` hoặc thêm điều kiện `v < best_val`.

### Lỗi 2: Trả về `0` thay vì `-1` khi không có số chẵn

In [23]:
def sai_mac_dinh_0(nums):
    best_val, best_cnt = 0, 0                # SAI: mặc định 0 (0 cũng là số chẵn hợp lệ!)
    for v, c in Counter(x for x in nums if x % 2 == 0).items():
        if c > best_cnt or (c == best_cnt and v < best_val):
            best_val, best_cnt = v, c
    return best_val

nums = [29, 47, 21]
print("đúng:", Solution().mostFrequentEven(nums), "| sai:", sai_mac_dinh_0(nums))
assert Solution().mostFrequentEven(nums) != sai_mac_dinh_0(nums)

đúng: -1 | sai: 0


**Vì sao sai:** khi không có số chẵn phải trả `-1`. Mặc định `0` vừa sai, vừa gây nhầm lẫn vì `0` là một đáp án hợp lệ khi `nums` chứa `0`. Hãy khởi tạo `best_val = -1`.

### Lỗi 3: Đếm cả số lẻ

In [24]:
def sai_dem_ca_le(nums):
    cnt = Counter(nums)                      # SAI: không lọc số chẵn
    best_val, best_cnt = -1, 0
    for v, c in cnt.items():
        if c > best_cnt or (c == best_cnt and v < best_val):
            best_val, best_cnt = v, c
    return best_val

nums = [1, 1, 1, 2]
print("đúng:", Solution().mostFrequentEven(nums), "| sai:", sai_dem_ca_le(nums))
assert Solution().mostFrequentEven(nums) != sai_dem_ca_le(nums)

đúng: 2 | sai: 1


**Vì sao sai:** số lẻ `1` xuất hiện nhiều nhất nên bị chọn, trong khi đề chỉ xét số **chẵn**. Luôn lọc trước bằng `x % 2 == 0`.

### Lỗi 4: Trả về tần suất thay vì giá trị

In [25]:
def sai_tra_tan_suat(nums):
    cnt = Counter(x for x in nums if x % 2 == 0)
    if not cnt:
        return -1
    return max(cnt.values())                 # SAI: trả số lần, không phải số

nums = [6, 6, 1]
print("đúng:", Solution().mostFrequentEven(nums), "| sai:", sai_tra_tan_suat(nums))
assert Solution().mostFrequentEven(nums) != sai_tra_tan_suat(nums)

đúng: 6 | sai: 2


**Vì sao sai:** đề yêu cầu trả về **phần tử**, không phải số lần xuất hiện của nó (ví dụ `[6, 6, 1]` trả `6`, không phải `2`).

### Lỗi 5: Dùng `Counter.most_common(1)` và tin nó phá hòa đúng

In [26]:
def sai_most_common(nums):
    cnt = Counter(x for x in nums if x % 2 == 0)
    if not cnt:
        return -1
    return cnt.most_common(1)[0][0]          # SAI: hòa -> trả phần tử gặp trước, không phải nhỏ nhất

nums = [4, 4, 2, 2]
print("đúng:", Solution().mostFrequentEven(nums), "| sai:", sai_most_common(nums))
assert Solution().mostFrequentEven(nums) != sai_most_common(nums)

đúng: 2 | sai: 4


**Vì sao sai:** `most_common` giữ thứ tự chèn khi hòa; ở đây `4` đến trước nên được trả, nhưng đáp án đúng là `2`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "phần tử xuất hiện nhiều nhất" ⇒ nghĩ ngay đến **đếm tần suất** (`Counter`).
- "hòa thì chọn nhỏ nhất / lớn nhất" ⇒ nghĩ đến khóa `(đếm, ∓giá trị)` hoặc điều kiện cập nhật có `c == best_cnt and v < best_val`.
- "không có thì trả `-1`" ⇒ khởi tạo đáp án là `-1` (đảm bảo không nhầm với đáp án hợp lệ `0`).
- Khi giá trị bị chặn nhỏ (ví dụ `<= 10^5`) có thể dùng mảng đếm; duyệt giá trị tăng dần sẽ phá hòa tự nhiên.
- Duyệt chỉ các số chẵn: `range(0, V + 1, 2)`.
- Luôn thử `[0]`, `[1]`, `[4, 4, 2, 2]`, và mảng toàn lẻ trước khi nộp.

## 15. Biến thể của bài

1. **Số lẻ phổ biến nhất**: đổi điều kiện lọc.
2. **Phần tử xuất hiện nhiều nhất, hòa chọn lớn nhất.**
3. **Trả về tất cả** phần tử chẵn đạt số lần lớn nhất (danh sách có thứ tự).
4. **Top-k phần tử chẵn** phổ biến nhất (dùng `heapq.nsmallest` với khóa `(-đếm, giá trị)`).
5. **Phần tử xuất hiện nhiều nhất theo chia hết cho `m`** (tổng quát hóa điều kiện chẵn).
6. **Dữ liệu dạng luồng** (stream): giữ `Counter` và `best` rồi cập nhật khi nhận phần tử mới.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute `nums.count` | với mỗi giá trị chẵn phân biệt, đếm lại cả mảng | `O(n^2)` | `O(k)` | `n` rất nhỏ, kiểm chứng |
| 2. `Counter` | đếm một lượt rồi chọn theo `(đếm, -giá trị)` | `O(n)` | `O(k)` | lựa chọn chuẩn khi làm bài |
| 3. Sắp xếp | sort số chẵn, đếm đoạn liên tiếp | `O(n log n)` | `O(n)` | khi không muốn dùng hash |
| 4. Mảng đếm | `cnt[v]` với `v <= 10^5`, duyệt tăng dần | `O(n + V)` | `O(V)` | giá trị bị chặn nhỏ, muốn phá hòa tự nhiên |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `Solution().mostFrequentEven([8, 8, 6, 6, 7, 7, 7])` là gì?

**Bài 2 (tính tay).** Với `nums = [10, 20, 20, 10, 30]`, lập bảng đếm số chẵn và chọn đáp án.

**Bài 3 (điền TODO).** Hoàn thành hàm `chon_dap_an(cnt)` nhận `dict` đếm các số chẵn, trả về đáp án.

In [27]:
def chon_dap_an(cnt):
    best_val, best_cnt = -1, 0
    for v, c in cnt.items():
        # TODO: cập nhật best_val, best_cnt theo quy tắc (đếm lớn hơn, hòa thì giá trị nhỏ hơn)
        pass
    return best_val

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `so_le_pho_bien_nhat(nums)`: số **lẻ** xuất hiện nhiều nhất, hòa thì nhỏ nhất, không có thì `-1`.

**Bài 5 (biến thể / mini challenge).** Viết `top_k_even(nums, k)` trả về danh sách tối đa `k` số chẵn phổ biến nhất, sắp theo `(đếm giảm dần, giá trị tăng dần)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: số lẻ `7` xuất hiện nhiều nhất nhưng bị bỏ. `8` và `6` hòa 2 lần. Bài 2: `10 -> 2`, `20 -> 2`, `30 -> 1`.
- **Mức 2 (vừa):** Bài 3: điều kiện `c > best_cnt or (c == best_cnt and v < best_val)`. Bài 4: đổi `x % 2 == 0` thành `x % 2 == 1`.
- **Mức 3 (sâu):** Bài 5: `sorted(cnt.items(), key=lambda p: (-p[1], p[0]))[:k]` rồi lấy khóa; chú ý `k` có thể lớn hơn số giá trị phân biệt.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
from collections import Counter

# Bài 1: 8 và 6 hòa (2 lần) -> nhỏ nhất là 6
assert Solution().mostFrequentEven([8, 8, 6, 6, 7, 7, 7]) == 6

# Bài 2: 10 và 20 hòa 2 lần -> 10
assert Solution().mostFrequentEven([10, 20, 20, 10, 30]) == 10

# Bài 3
def chon_dap_an(cnt):
    best_val, best_cnt = -1, 0
    for v, c in cnt.items():
        if c > best_cnt or (c == best_cnt and v < best_val):
            best_val, best_cnt = v, c
    return best_val

# Bài 4
def so_le_pho_bien_nhat(nums):
    cnt = Counter(x for x in nums if x % 2 == 1)
    best_val, best_cnt = -1, 0
    for v, c in cnt.items():
        if c > best_cnt or (c == best_cnt and v < best_val):
            best_val, best_cnt = v, c
    return best_val

# Bài 5
def top_k_even(nums, k):
    cnt = Counter(x for x in nums if x % 2 == 0)
    ranked = sorted(cnt.items(), key=lambda p: (-p[1], p[0]))
    return [v for v, _ in ranked[:k]]

assert chon_dap_an({0: 1, 2: 2, 4: 2}) == 2
assert chon_dap_an({}) == -1
assert chon_dap_an({6: 3}) == 6
assert so_le_pho_bien_nhat([8, 8, 6, 6, 7, 7, 7]) == 7
assert so_le_pho_bien_nhat([2, 4, 6]) == -1
assert so_le_pho_bien_nhat([1, 1, 3, 3]) == 1
assert top_k_even([0, 1, 2, 2, 4, 4, 1], 2) == [2, 4]
assert top_k_even([0, 1, 2, 2, 4, 4, 1], 10) == [2, 4, 0]
assert top_k_even([1, 3], 3) == []
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đếm có lọc + phá hòa)

```python
class Solution:
    def mostFrequentEven(self, nums: List[int]) -> int:
        cnt = Counter(x for x in nums if x % 2 == 0)
        best_val, best_cnt = -1, 0
        for v, c in cnt.items():
            if c > best_cnt or (c == best_cnt and v < best_val):
                best_val, best_cnt = v, c
        return best_val
```

**Checklist**

- [ ] Đã lọc **chỉ số chẵn** (`x % 2 == 0`), nhớ `0` là chẵn?
- [ ] Khởi tạo `best_val = -1`, `best_cnt = 0` để không có số chẵn thì trả `-1`?
- [ ] Quy tắc phá hòa: đếm lớn hơn, hòa thì **giá trị nhỏ hơn**?
- [ ] Trả về **giá trị**, không phải số lần xuất hiện?
- [ ] Không dùng `most_common(1)` để phá hòa?
- [ ] Đã thử `[0]`, `[1]`, `[4,4,2,2]`, và mảng toàn số lẻ?
- [ ] Signature đúng `mostFrequentEven(self, nums: List[int]) -> int` và có `from typing import List`?